<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a2e 35%, #2d1b69 70%, #0d3b2e 100%);padding: 60px 40px; border-radius: 18px; text-align: center; margin-bottom: 10px; border: 1px solid #7b2fff44;"><h1 style="color: #7b2fff; font-family: 'Courier New', monospace; font-size: 2.9em;letter-spacing: 5px; margin: 0 0 12px 0; text-shadow: 0 0 40px #7b2fff66;">🖼️ GÖRÜNTÜ GEOMETRİSİ</h1><h2 style="color: #a8dadc; font-family: 'Courier New', monospace;font-size: 1.35em; margin: 0 0 18px 0; font-weight: 300;">8. Hafta — Boyutlandırma, Çevirme & Döndürme</h2><p style="color: #6b7280; font-family: 'Courier New', monospace; font-size: 0.85em; margin: 0;">Interpolasyon &bull; resize() &bull; flip() &bull; rotate() &bull; Affine Dönüşüm &bull; warpAffine()</p></div>

---
# 📐 BÖLÜM 1: Geometrik Görüntü Dönüşümleri — Giriş

Geometrik dönüşümler, bir görüntüdeki **piksellerin konumunu değiştiren** işlemlerdir. Boyutlandırma, döndürme, aynalama, kaydırma ve perspektif düzeltme bunların başında gelir.

Matematiksel temel: Her geometrik dönüşüm bir **matris çarpımı** ile ifade edilebilir.

```
[x']   [a  b  tx]   [x]
[y'] = [c  d  ty] × [y]
[1 ]   [0  0   1]   [1]
```
Bu 3×3 matrise **Affine Dönüşüm Matrisi** denir.

<div style="background: #2d1800; border-left: 5px solid #f4a261; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f4a261; font-size: 1.08em;">📜 Tarihçe</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Dijital Görüntü Geometrisinin Kökenleri:</b><br><br>Geometrik görüntü dönüşümleri, 1960'larda NASA'nın uzay fotoğraflarını düzeltmesiyle bilgisayar bilimine girdi. Ay ve Mars'a gönderilen sondaların çektiği görüntüler mercek bozukluğu, kamera açısı ve gezegen yüzeyi eğriliklerinden dolayı çarpıktı.<br><br>1964'te NASA'nın <b>Ranger 7</b> uydusunun Ay fotoğraflarını düzeltmek için ilk dijital görüntü geometrisi algoritmaları geliştirildi. Bugün milyarlarca kez çalışan <code>resize()</code> fonksiyonunun temeli bu uzay programlarına dayanır.<br><br><b>İlginç:</b> 1968'de Kubrick'in "2001: A Space Odyssey" filminde kullanılan bazı uzay efektleri, NASA'nın bu geometri düzeltme tekniklerinden ilham almıştır!</span></div>

## 1.1 Neden Geometrik Dönüşüm Gerekir?

| Uygulama | Kullanılan Dönüşüm |
|----------|--------------------|
| Sosyal medya profil fotoğrafı kırpma | resize + crop |
| Instagram filtre aynalama | flip |
| Google Maps yol görüntüsü düzeltme | perspektif dönüşüm |
| Tıbbi görüntü hizalama | affine dönüşüm |
| Otonom araç kamera kalibrasyonu | distortion correction |
| Barkod/QR kod okuma (eğik tutulmuş) | affine + perspektif |
| Yüz hizalama (face alignment) | rotate + scale |
| Veri artırma (data augmentation) | tümü birlikte |

---
# 🔍 BÖLÜM 2: İnterpolasyon — En Kritik Kavram

Bir görüntüyü yeniden boyutlandırdığınızda eski piksellerin tam arasına düşen yeni piksellerin değerleri **nereden gelir?** İşte bu sorunun cevabı **interpolasyondur**.

```
Orijinal 2×2 piksel:        4×4'e büyütülünce aralar ne olur?

  [100] [200]          [100] [?]  [?]  [200]
  [150] [250]   →      [?]   [?]  [?]  [?]
                        [?]   [?]  [?]  [?]
                        [150] [?]  [?]  [250]
```

**? değerlerini doldurmak için farklı stratejiler = farklı interpolasyon yöntemleri**

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 İnterpolasyonun Matematiği</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>En Yakın Komşu (INTER_NEAREST):</b> En basit yöntem. Yeni piksel, kendisine en yakın eski pikselin değerini alır. Hızlıdır ama "pikselleşme" etkisi yaratır. Piksel sanatı için kasıtlı kullanılır!<br><br><b>Bilineer (INTER_LINEAR):</b> 4 komşu pikselin ağırlıklı ortalaması. <code>f(x,y) ≈ ax + by + cxy + d</code> formülüyle hesaplanır. Hız/kalite dengesi en iyi olan yöntem.<br><br><b>Bikübik (INTER_CUBIC):</b> 16 komşu pikseli kullanır. Kenarlar daha pürüzsüz. Bilineerin iki katı işlem gücü gerektirir ama gözle görülür kalite artışı sağlar.<br><br><b>Lanczos (INTER_LANCZOS4):</b> 64 komşu piksel! Sinyalleri teorik olarak mükemmel reconstruct eden sinc fonksiyonu tabanlı. Claude Shannon'ın <b>örnekleme teoreminden</b> türetilmiştir (1949). En kaliteli ama en yavaş yöntem.<br><br><b>Alan (INTER_AREA):</b> Küçültme için özel. Gerçek piksel alanlarının ortalaması. Küçültmede Lanczos'tan bile iyi sonuç verir, büyütmede işe yaramaz.</span></div>

## 2.1 İnterpolasyon Yöntemleri Özeti

| Yöntem | Komşu Piksel | Hız | Kalite | En İyi Kullanım |
|--------|-------------|-----|--------|-----------------|
| `INTER_NEAREST` | 1 | ⭐⭐⭐⭐⭐ | ⭐ | Maske/label görüntüleri, piksel sanatı |
| `INTER_LINEAR` | 4 | ⭐⭐⭐⭐ | ⭐⭐⭐ | Genel amaç, varsayılan |
| `INTER_CUBIC` | 16 | ⭐⭐⭐ | ⭐⭐⭐⭐ | Fotoğraf büyütme |
| `INTER_LANCZOS4` | 64 | ⭐⭐ | ⭐⭐⭐⭐⭐ | Baskı kalitesi, arşiv |
| `INTER_AREA` | değişken | ⭐⭐⭐ | ⭐⭐⭐⭐⭐ | **Yalnızca küçültme** |

> **Altın kural:** Büyütme için `INTER_CUBIC` veya `INTER_LANCZOS4`, küçültme için `INTER_AREA` kullanın.

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Neden INTER_NEAREST maske için zorunlu?</b><br><br>Derin öğrenme segmentasyon görevlerinde görüntü ve onun etiketi (mask) birlikte resize edilir. Maske pikselleri tam sayı sınıf ID'lerdir: 0=arka plan, 1=araba, 2=insan...<br><br>Eğer maskeye INTER_LINEAR uygularsak araba(1) ve insan(2) sınırındaki piksel <code>(1+2)/2 = 1.5</code> olur — bu geçersiz bir sınıf! INTER_NEAREST ise ya 1 ya 2 döndürür, hiçbir zaman ara değer üretmez.<br><br>PyTorch ve TensorFlow'un veri pipeline'larında mask resize her zaman NEAREST kullanır.</span></div>

---
# 💻 BÖLÜM 3: Kodlar

## 3.1 Temel `cv.resize()` Kullanımı

```python
cv.resize(kaynak, (genişlik, yükseklik), interpolation=yöntem)
```

⚠️ **Sık yapılan hata:** NumPy'da `shape` **(yükseklik, genişlik)** sırasındayken `cv.resize()` hedef boyutu **(genişlik, yükseklik)** olarak alır — tam tersi!

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/bugday.jpg")

print("Orijinal boyut:", resim.shape)  # (yükseklik, genişlik, kanal)

# Sabit boyuta getirme
yeni_resim = cv.resize(resim, (300, 300))
print("Yeni boyut:", yeni_resim.shape)

cv.imshow("Orijinal", resim)
cv.imshow("300x300", yeni_resim)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #2d2200; border-left: 5px solid #f9c74f; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f9c74f; font-size: 1.08em;">⚠️ Dikkat!</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><code>cv.resize(resim, (300, 300))</code> — buradaki <code>(300, 300)</code> <b>(genişlik, yükseklik)</b> sırasındadır!<br><br>Ama <code>resim.shape</code> <b>(yükseklik, genişlik, kanal)</b> sırasındadır.<br><br>Dolayısıyla: <code>yukseklik, genislik = resim.shape[:2]</code> alıp <code>cv.resize(resim, (genislik, yukseklik))</code> yazmanız gerekir. Bu en yaygın OpenCV hatalarından biridir!</span></div>

## 3.2 İnterpolasyon Yöntemi Belirterek Büyütme

In [ ]:
import cv2 as cv

resim = cv.imread("resim/sudoku.jpg")
print("Orijinal:", resim.shape)

# Bicubic ile büyütme — fotoğraf büyütmede en yaygın
yeni_resim = cv.resize(resim, (2000, 2000), interpolation=cv.INTER_CUBIC)
print("Yeni boyut:", yeni_resim.shape)

cv.imwrite("resim/sudoku_2000_2000.jpg", yeni_resim)
cv.imshow("a", yeni_resim)
cv.waitKey(0)
cv.destroyAllWindows()

## 3.3 Alternatif Küçültme — NumPy Dilimleme

`resize()` kullanmadan görüntü küçültmenin hızlı ama kaba bir yolu: NumPy'ın adım (step) dilimleme sözdizimi.

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/bugday.jpg")
print("Orijinal:", resim.shape)

# Her 4. pikseli al → 1/4 boyuta küçülür
yeni_resim = resim[::4, ::4]
print("Küçültülmüş (::4):", yeni_resim.shape)

# Her 2. pikseli al → 1/2 boyuta küçülür
yarisi = resim[::2, ::2]
print("Yarı boyut (::2):", yarisi.shape)

cv.imshow("Orijinal", resim)
cv.imshow("1/4 boyut (::4)", yeni_resim)
cv.imshow("1/2 boyut (::2)", yarisi)
cv.waitKey(0)
cv.destroyAllWindows()

<div style="background: #0d2200; border-left: 5px solid #90be6d; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #90be6d; font-size: 1.08em;">🔄 Karşılaştırma</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>NumPy dilimleme vs cv.resize() — Ne zaman hangisi?</b><br><br><b>NumPy [::n, ::n]</b><br>✅ Son derece hızlı (sıfır hesaplama, yalnızca bellek adresleme)<br>✅ Orijinal piksel değerleri korunur (interpolasyon yok)<br>❌ Yalnızca tam sayı kat küçültme (2x, 3x, 4x...)<br>❌ Kenar yumuşatma yok → ince çizgiler kaybolabilir (aliasing)<br><br><b>cv.resize()</b><br>✅ Herhangi bir hedef boyut<br>✅ İnterpolasyon → daha kaliteli sonuç<br>❌ Hesaplama maliyeti var<br><br><b>Sonuç:</b> Prototipleme ve hız kritikse NumPy, kalite önemliyse <code>resize(INTER_AREA)</code>.</span></div>

## 3.4 Görüntü Büyütme — Alternatif Yol

`resize()` kullanmadan büyütme: Her pikseli `np.repeat()` ile kopyalamak.

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/bugday.jpg")
print("Orijinal:", resim.shape)

# 2 katına büyütme — her pikseli hem yatay hem dikey 2 kez tekrar et
buyutulmus = np.repeat(np.repeat(resim, 2, axis=0), 2, axis=1)
print("2x büyütülmüş:", buyutulmus.shape)

# Karşılaştırma: resize ile aynı boyuta getir
resim_resize = cv.resize(resim, (resim.shape[1]*2, resim.shape[0]*2), interpolation=cv.INTER_CUBIC)
print("resize 2x:", resim_resize.shape)

cv.imshow("Orijinal", resim)
cv.imshow("np.repeat 2x (NEAREST benzeri)", buyutulmus)
cv.imshow("cv.resize CUBIC 2x", resim_resize)
cv.waitKey(0)
cv.destroyAllWindows()

## 3.5 Tüm İnterpolasyon Yöntemlerini Karşılaştırma

matplotlib ile yan yana görselleştirme.

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt

resim = cv.imread("resim/sudoku.jpg")

yontemler = [
    (cv.INTER_NEAREST,  "INTER_NEAREST\n(En Yakın)"),
    (cv.INTER_LINEAR,   "INTER_LINEAR\n(Bilineer)"),
    (cv.INTER_CUBIC,    "INTER_CUBIC\n(Bikübik)"),
    (cv.INTER_LANCZOS4, "INTER_LANCZOS4\n(Lanczos)"),
    (cv.INTER_AREA,     "INTER_AREA\n(Alan — büyütmede zayıf)"),
]

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
fig.suptitle("İnterpolasyon Yöntemleri Karşılaştırması — 2000×2000'e Büyütme", fontsize=13)

# Orijinal
axes[0, 0].imshow(cv.cvtColor(resim, cv.COLOR_BGR2RGB))
axes[0, 0].set_title(f"Orijinal\n{resim.shape[1]}×{resim.shape[0]}", fontsize=9)
axes[0, 0].axis("off")

for idx, (yontem, baslik) in enumerate(yontemler):
    buyutulmus = cv.resize(resim, (2000, 2000), interpolation=yontem)
    ax = axes[(idx+1)//3][(idx+1)%3]
    ax.imshow(cv.cvtColor(buyutulmus, cv.COLOR_BGR2RGB))
    ax.set_title(baslik, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Neden Lanczos en iyi görünüyor?</b><br><br>Lanczos filtresi, Claude Shannon'ın 1949 tarihli örnekleme teoreminin mükemmel uygulamasıdır. Teorem der ki: "Bir sinyali tam olarak yeniden oluşturmak için en az 2× örnekleme frekansı gerekir." (Nyquist-Shannon teoremi)<br><br>Lanczos bu teoremi görüntülere uygular: sinc(x) = sin(πx)/(πx) fonksiyonu, teorik olarak sınırlı bir bant genişliğindeki tüm bilgiyi korur. 64 komşu pikseli hesaba katar — bu neden yavaş ama mükemmel sonuç verdiğini açıklar.<br><br><b>Gerçek dünya uygulaması:</b> Adobe Photoshop'un "Bicubic Sharper" seçeneği Lanczos türevi bir algoritma kullanır. Profesyonel baskı evleri ve film stüdyoları görüntü ölçekleme için Lanczos'u standart kabul eder.</span></div>

## 3.6 Toplu Görüntü İşleme — Batch Resize

Gerçek projelerde yüzlerce/binlerce görüntüyü aynı anda işlemek gerekir. `glob`, `os` ve `pathlib` ile otomatik pipeline.

In [ ]:
import cv2 as cv
import os
import glob
from pathlib import Path

"""
Resim klasörünün altındaki test klasöründe bulunan 1024×1024 boyutundaki resimleri
test klasörünün altında ufak isminde bir klasör açarak
resimleri de 512×512 boyutuna getirerek aynı isimlerle kaydeden program
"""

# Hedef klasörü oluştur (yoksa)
yol = "resim/test/ufak"
os.makedirs(yol, exist_ok=True)  # exist_ok=True: zaten varsa hata verme

# Klasördeki tüm jpg dosyaları
resimler = glob.glob("resim/test/*.jpg")
print(f"Toplam {len(resimler)} resim bulundu")

basarili = 0
for resim_yolu in resimler:
    # Oku
    goruntu = cv.imread(resim_yolu)
    if goruntu is None:
        print(f"UYARI: {resim_yolu} okunamadı, atlanıyor")
        continue

    # 512×512'ye küçült — küçültme için INTER_AREA en kaliteli
    kucultulmus = cv.resize(goruntu, (512, 512), interpolation=cv.INTER_AREA)

    # Yalnızca dosya adını al (uzantısız)
    dosya_adi = Path(resim_yolu).stem

    # Kaydet
    hedef = f"{yol}/{dosya_adi}.jpg"
    cv.imwrite(hedef, kucultulmus)
    basarili += 1
    print(f"[{basarili:3d}] {resim_yolu} → {hedef}")

print(f"\nTamamlandı: {basarili}/{len(resimler)} resim işlendi")

<div style="background: #2d0020; border-left: 5px solid #f72585; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #f72585; font-size: 1.08em;">🌍 Gerçek Dünya</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>Veri Artırma (Data Augmentation) — Derin Öğrenmenin Gizli Silahı</b><br><br>Bu toplu resize işlemi, makine öğrenmesinde "veri ön işleme pipeline'ı"nın temelidir. ImageNet veri seti (14 milyon görüntü) her eğitimden önce 224×224 veya 256×256'ya resize edilir. ResNet, VGG, EfficientNet gibi modeller hep bu şekilde hazırlanmış verilerle eğitilir.<br><br>Ama sadece resize değil: her eğitimde görüntüler rastgele döndürülür, çevrilir, kırpılır ve parlaklığı değiştirilir. Bu sayede 50.000 görüntüden milyonlarca farklı eğitim örneği üretilir — modelin ezberlemesi önlenir.<br><br>PyTorch'ta <code>transforms.Compose([Resize(), RandomHorizontalFlip(), RandomRotation(30)...])</code> tam da bu hafta öğrendiğiniz işlemleri yapar!</span></div>

---
# 🪞 BÖLÜM 4: Aynalama — `cv.flip()`

`cv.flip(görüntü, flipCode)` fonksiyonu görüntüyü aynalar.

| flipCode | Efekt | Açıklama |
|----------|-------|----------|
| `0` | ↕ | Yatay eksen etrafında (yukarı-aşağı) |
| `1` | ↔ | Dikey eksen etrafında (sağ-sol) |
| `-1` | ↕↔ | Her iki eksen etrafında (180° dönüş gibi) |

```
Orijinal:    flip(0):    flip(1):    flip(-1):
A B C        G H I        C B A        I H G
D E F   →   D E F   →   F E D   →   F E D
G H I        A B C        I H G        C B A
```

In [ ]:
import cv2 as cv

resim = cv.imread("resim/sudoku.jpg")

# flipCode = -1: yatay + dikey aynalama (180° dönüşe eşdeğer)
yeni_resim = cv.flip(resim, -1)

cv.imshow("Orijinal", resim)
cv.imshow("flip(-1)", yeni_resim)
cv.waitKey(0)
cv.destroyAllWindows()

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt

resim = cv.imread("resim/sudoku.jpg")

sonuclar = [
    (resim,              "Orijinal"),
    (cv.flip(resim, -1), "flip(-1) — her iki eksen"),
    (cv.flip(resim,  0), "flip(0)  — yukarı/aşağı"),
    (cv.flip(resim,  1), "flip(1)  — sağ/sol"),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle("cv.flip() — Aynalama Modları", fontsize=13)

for ax, (goruntu, baslik) in zip(axes.flat, sonuclar):
    ax.imshow(cv.cvtColor(goruntu, cv.COLOR_BGR2RGB))
    ax.set_title(baslik, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 flip() Neden Önemli? — Veri Artırmada Merkezi Rol</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">Makine öğrenmesinde <b>yatay flip en yaygın augmentation tekniğidir</b>. Nedeni şu: bir kedi fotoğrafını aynalamanız, hâlâ bir kedi fotoğrafı yaratır — etiket değişmez ama model daha fazla çeşitlilik görür.<br><br>Ancak dikkatli olunmalı: bazı sınıflar için flip geçersizdir!<br>• Trafik levhası ("DUR" aynalayınca anlamsız) → flip kullanma<br>• El yazısı rakamlar (6↔9, flip'te farklı anlam) → dikkatli ol<br>• Tıbbi görüntü (karaciğer sağda, kalp solda — flip anatomik bozukluk yaratır) → özel durum<br><br><b>Dikey flip</b> daha az kullanılır — çünkü doğal fotoğraflarda nesneler nadiren baş aşağı durur. Uydu/hava görüntülerinde ise dikey flip de mantıklıdır.</span></div>

---
# 🔄 BÖLÜM 5: Döndürme — `cv.rotate()` ve `cv.warpAffine()`

## 5.1 Sabit Açılarla Döndürme — `cv.rotate()`

`cv.rotate()` yalnızca **90°, 180°, 270°** döndürme yapar. Hızlıdır çünkü satır/sütun transpozisyonu yeterlidir — matematik hesaplama gerekmez.

| Sabit | Açı |
|-------|-----|
| `cv.ROTATE_90_CLOCKWISE` | 90° saat yönünde |
| `cv.ROTATE_180` | 180° |
| `cv.ROTATE_90_COUNTERCLOCKWISE` | 90° saat yönü tersine (= 270° saat yönünde) |

In [ ]:
import cv2 as cv
import matplotlib.pyplot as plt

resim = cv.imread("resim/sudoku.jpg")

sonuclar = [
    (resim, "Orijinal"),
    (cv.rotate(resim, cv.ROTATE_90_CLOCKWISE),      "90° Saat Yönünde"),
    (cv.rotate(resim, cv.ROTATE_180),               "180°"),
    (cv.rotate(resim, cv.ROTATE_90_COUNTERCLOCKWISE),"90° Saat Yönü Tersine"),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle("cv.rotate() — Sabit Açı Döndürme", fontsize=13)

for ax, (g, b) in zip(axes.flat, sonuclar):
    ax.imshow(cv.cvtColor(g, cv.COLOR_BGR2RGB))
    ax.set_title(b, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 5.2 Serbest Açıyla Döndürme — Affine Dönüşüm

Herhangi bir açıyla döndürme için **Affine Dönüşüm** kullanılır. İki adım:

1. `cv.getRotationMatrix2D(merkez, açı, ölçek)` → 2×3 dönüşüm matrisi
2. `cv.warpAffine(görüntü, matris, boyut)` → dönüşümü uygula

```
Rotasyon Matrisi:
[ cos(θ)  -sin(θ)  tx ]
[ sin(θ)   cos(θ)  ty ]
```

`tx` ve `ty` : döndürme sonrası görüntünün çerçeve içinde kalması için gerekli öteleme miktarı.

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

resim = cv.imread("resim/sudoku.jpg")

def rotate_image(image, angle):
    """Görüntüyü merkezi etrafında verilen açıyla döndürür."""
    merkez = tuple(np.array(image.shape[1::-1]) / 2)  # (genişlik/2, yükseklik/2)
    rot_mat = cv.getRotationMatrix2D(merkez, angle, 1.0)  # ölçek=1.0 (boyut değişmez)
    result = cv.warpAffine(image, rot_mat, image.shape[1::-1], flags=cv.INTER_LINEAR)
    return result

sonuclar = [
    (resim,                  "Orijinal"),
    (rotate_image(resim,  45), "45° döndürme"),
    (rotate_image(resim, 135), "135° döndürme"),
    (rotate_image(resim, 215), "215° döndürme"),
]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
fig.suptitle("warpAffine — Serbest Açı Döndürme", fontsize=13)

for ax, (g, b) in zip(axes.flat, sonuclar):
    ax.imshow(cv.cvtColor(g, cv.COLOR_BGR2RGB))
    ax.set_title(b, fontsize=9)
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import cv2 as cv
import numpy as np

resim = cv.imread("resim/sudoku.jpg")
h, w = resim.shape[:2]
merkez = (w // 2, h // 2)

# Dönüşüm matrisini inceleyelim
rot_mat = cv.getRotationMatrix2D(merkez, 45, 1.0)
print("Rotasyon Matrisi (45°):")
print(rot_mat)
print(f"\ncos(45°) = {np.cos(np.radians(45)):.4f}")
print(f"sin(45°) = {np.sin(np.radians(45)):.4f}")
print(f"Matristeki [0,0] = {rot_mat[0,0]:.4f}  ← cos(45°)")
print(f"Matristeki [0,1] = {rot_mat[0,1]:.4f}  ← -sin(45°)")
print(f"Matristeki [1,0] = {rot_mat[1,0]:.4f}  ← sin(45°)")
print(f"Matristeki [0,2] = {rot_mat[0,2]:.4f}  ← yatay öteleme (tx)")
print(f"Matristeki [1,2] = {rot_mat[1,2]:.4f}  ← dikey öteleme (ty)")

<div style="background: #001e2d; border-left: 5px solid #00b4d8; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #00b4d8; font-size: 1.08em;">🔬 Teknik Derinlik</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;"><b>warpAffine — Affine Dönüşümün Gücü</b><br><br><code>cv.warpAffine()</code> aslında çok daha genel bir işlem yapabilir. 2×3 matrisin farklı değerleri farklı dönüşümleri temsil eder:<br><br><table style="color:#cdd6f4; border-collapse: collapse; width:100%;"><tr style="color:#00ff88"><th>Dönüşüm</th><th>Matris</th></tr><tr><td>Öteleme (Translation)</td><td>[[1,0,tx],[0,1,ty]]</td></tr><tr><td>Ölçekleme (Scale)</td><td>[[sx,0,0],[0,sy,0]]</td></tr><tr><td>Döndürme (Rotation)</td><td>[[cos θ, -sin θ, tx],[sin θ, cos θ, ty]]</td></tr><tr><td>Yatay kesme (Shear)</td><td>[[1, k, 0],[0, 1, 0]]</td></tr><tr><td>Hepsi birlikte</td><td>[[a,b,tx],[c,d,ty]]</td></tr></table><br>Bu matris, kağıt üzerinde düz kalırken koordinatları dönüştürür. Paralel çizgiler affine dönüşümde hep paralel kalır — perspektif dönüşümden farkı budur!</span></div>

<div style="background: #1a0d2e; border-left: 5px solid #c77dff; padding: 18px 24px; border-radius: 0 10px 10px 0; margin: 14px 0;"><strong style="color: #c77dff; font-size: 1.08em;">🧠 Görüntü Döndürmenin Dijital Sinema Tarihi</strong><br><br><span style="color: #cdd6f4; line-height: 1.9;">2000'li yıllara kadar profesyonel sinema kameraları yanlış açıda çekilmiş sahneleri düzeltmek için fiziksel optik filtreler kullanıyordu. Dijital post-prodüksiyon devrimiyle bu işlem yazılıma taşındı.<br><br>Bugün Netflix ve Prime Video gibi platformların içerik kalite kontrol pipeline'larında otomatik görüntü hizalama (auto-deskew) çalışır: eğik çekilen sahneler, yatay çizgilerin tespiti ve Hough Transform + warpAffine kombinasyonuyla otomatik düzeltilir.<br><br><b>İlginç:</b> Instagram'ın "Auto-Level" özelliği, ufuk çizgisini bulup fotoğrafı döndüren tam da bu tekniktir. Bir selfie'yi doğrultmak için saniyede milyonlarca matris çarpımı yapılır!</span></div>

---
# 🚀 BÖLÜM 6: Gelişmiş Uygulamalar

## 6.1 Kırpma Kayıpsız Döndürme

Standart `warpAffine` ile döndürünce köşeler siyah kalır. Aşağıdaki fonksiyon köşelerin kaybolmaması için canvas'ı otomatik büyütür.

In [ ]:
import cv2 as cv
import numpy as np

def rotate_full(image, angle):
    """
    Kırpma olmadan döndürme: Canvas görüntünün tamamını sığdıracak kadar büyür.
    """
    h, w = image.shape[:2]
    cx, cy = w / 2, h / 2

    # Dönüşüm matrisi
    M = cv.getRotationMatrix2D((cx, cy), angle, 1.0)

    # Yeni canvas boyutunu hesapla
    cos_a = abs(M[0, 0])
    sin_a = abs(M[0, 1])
    yeni_w = int(h * sin_a + w * cos_a)
    yeni_h = int(h * cos_a + w * sin_a)

    # Merkezi yeni canvas'a taşı
    M[0, 2] += (yeni_w / 2) - cx
    M[1, 2] += (yeni_h / 2) - cy

    return cv.warpAffine(image, M, (yeni_w, yeni_h))

resim = cv.imread("resim/sudoku.jpg")

standart = cv.warpAffine(
    resim,
    cv.getRotationMatrix2D(
        (resim.shape[1]//2, resim.shape[0]//2), 30, 1.0
    ),
    (resim.shape[1], resim.shape[0])
)

tam = rotate_full(resim, 30)

print("Standart rotate boyutu:", standart.shape)
print("Kırpmasız rotate boyutu:", tam.shape)

cv.imshow("Standart (30°) — köşeler kayıp", standart)
cv.imshow("Kırpmasız (30°) — tüm piksel korundu", tam)
cv.waitKey(0)
cv.destroyAllWindows()

## 6.2 Orantılı Yeniden Boyutlandırma

En, boy oranını koruyarak yalnızca maksimum boyutu belirlemek.

In [ ]:
import cv2 as cv

def orantili_resize(image, max_boyut=800):
    """
    En-boy oranını koruyarak görüntüyü max_boyut piksel sınırına sığdırır.
    Hem yatay hem dikey görüntülerde doğru çalışır.
    """
    h, w = image.shape[:2]
    oran = min(max_boyut / w, max_boyut / h)  # küçüklük belirleyen boyut esas alınır

    yeni_w = int(w * oran)
    yeni_h = int(h * oran)

    # Büyütme mi küçültme mi? — doğru interpolasyon seç
    if oran < 1:
        interpolation = cv.INTER_AREA    # küçültme için en iyi
    else:
        interpolation = cv.INTER_CUBIC  # büyütme için iyi

    return cv.resize(image, (yeni_w, yeni_h), interpolation=interpolation)

resim = cv.imread("resim/bugday.jpg")
print(f"Orijinal: {resim.shape[1]}×{resim.shape[0]}")

duzenlenmis = orantili_resize(resim, max_boyut=400)
print(f"400px sınırlı: {duzenlenmis.shape[1]}×{duzenlenmis.shape[0]}")

cv.imshow("Orantılı resize", duzenlenmis)
cv.waitKey(0)
cv.destroyAllWindows()

---
# 📊 BÖLÜM 7: Tüm Operasyonların Özet Karşılaştırması

In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

# Test görüntüsü oluştur (harfler içeren)
test = np.ones((200, 300, 3), dtype=np.uint8) * 240
cv.putText(test, "OPENCV", (30, 80), cv.FONT_HERSHEY_DUPLEX, 1.5, (30, 30, 180), 2)
cv.putText(test, "8.Hafta", (50, 140), cv.FONT_HERSHEY_SIMPLEX, 1.0, (180, 30, 30), 2)
cv.rectangle(test, (5, 5), (295, 195), (0, 150, 0), 2)

def rot(img, a):
    cx, cy = img.shape[1]//2, img.shape[0]//2
    M = cv.getRotationMatrix2D((cx, cy), a, 1.0)
    return cv.warpAffine(img, M, (img.shape[1], img.shape[0]))

ops = [
    (test,                                          "Orijinal"),
    (cv.resize(test, (150, 100)),                   "resize küçük\n(150×100)"),
    (cv.resize(test, (450, 300), interpolation=cv.INTER_CUBIC), "resize büyük\nCUBIC"),
    (cv.resize(test, (450, 300), interpolation=cv.INTER_NEAREST),"resize büyük\nNEAREST"),
    (cv.flip(test, 1),                              "flip(1)\nSağ-Sol"),
    (cv.flip(test, 0),                              "flip(0)\nYukarı-Aşağı"),
    (cv.flip(test, -1),                             "flip(-1)\nHer İkisi"),
    (cv.rotate(test, cv.ROTATE_90_CLOCKWISE),       "rotate 90° CW"),
    (rot(test, 30),                                 "warpAffine\n30°"),
]

fig, axes = plt.subplots(3, 3, figsize=(14, 11))
fig.suptitle("resize | flip | rotate — Özet Karşılaştırma", fontsize=14, fontweight="bold")

for ax, (g, b) in zip(axes.flat, ops):
    ax.imshow(cv.cvtColor(g, cv.COLOR_BGR2RGB))
    ax.set_title(b, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()

---
# 🎓 BÖLÜM 8: Web Formu ile Görüntü İşleme Uygulaması

## Proje Açıklaması

Bir web formu oluşturun. Kullanıcı resim yüklesin, boyut/döndürme/çevirme/kırpma işlemlerini ayarlasın, işlenmiş resmi görsün.

**Teknoloji Seçenekleri:**

| Katman | Seçenek 1 | Seçenek 2 | Seçenek 3 |
|--------|-----------|-----------|----------|
| Ön yüz | HTML/CSS/JS | React | Vue.js |
| Arka yüz | Flask (Python) | PHP | FastAPI |
| Görüntü | OpenCV | Pillow + OpenCV | OpenCV |

**Flask + OpenCV mimarisi:**
```
Tarayıcı ──POST──→ Flask /upload → OpenCV işle → Base64 → JSON → Tarayıcı
```

In [ ]:
# Flask + OpenCV mini proje iskeleti (çalışan örnek)

flask_kodu = """
from flask import Flask, request, jsonify, render_template_string
import cv2
import numpy as np
import base64

app = Flask(__name__)

HTML = """\n<!DOCTYPE html>
<html>
<head><title>Görüntü İşleyici</title></head>
<body>
  <h2>OpenCV Web Görüntü İşleyici</h2>
  <form id="frm">
    <input type="file" id="dosya" accept="image/*"><br><br>
    Genişlik: <input type="number" id="gw" value="300"><br>
    Yükseklik: <input type="number" id="gh" value="300"><br>
    Döndür (°): <input type="number" id="angle" value="0"><br>
    Çevir: <select id="flip">
      <option value="none">Yok</option>
      <option value="0">Yukarı/Aşağı</option>
      <option value="1">Sağ/Sol</option>
      <option value="-1">Her İkisi</option>
    </select><br><br>
    <button type="button" onclick="isle()">İşle!</button>
  </form>
  <img id="sonuc" style="max-width:600px;margin-top:20px">
  <script>
    async function isle() {
      const f = document.getElementById("dosya").files[0];
      const fd = new FormData();
      fd.append("file", f);
      fd.append("gw", document.getElementById("gw").value);
      fd.append("gh", document.getElementById("gh").value);
      fd.append("angle", document.getElementById("angle").value);
      fd.append("flip", document.getElementById("flip").value);
      const r = await fetch("/isle", {method:"POST", body:fd});
      const d = await r.json();
      document.getElementById("sonuc").src = "data:image/jpeg;base64," + d.img;
    }
  </script>
</body></html>\n"""

@app.route("/")
def index():
    return render_template_string(HTML)

@app.route("/isle", methods=["POST"])
def isle():
    # Dosyayı oku
    f = request.files["file"].read()
    img = cv2.imdecode(np.frombuffer(f, np.uint8), cv2.IMREAD_COLOR)

    # Parametreler
    gw    = int(request.form.get("gw", 300))
    gh    = int(request.form.get("gh", 300))
    angle = float(request.form.get("angle", 0))
    flip  = request.form.get("flip", "none")

    # İşlemler
    img = cv2.resize(img, (gw, gh), interpolation=cv2.INTER_CUBIC)

    if angle != 0:
        cx, cy = gw // 2, gh // 2
        M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
        img = cv2.warpAffine(img, M, (gw, gh))

    if flip != "none":
        img = cv2.flip(img, int(flip))

    # Base64'e çevir
    _, buf = cv2.imencode(".jpg", img)
    b64 = base64.b64encode(buf).decode()
    return jsonify({"img": b64})

if __name__ == "__main__":
    app.run(debug=True)
"""

print("Flask uygulaması iskeleti hazırlandı.")
print("Çalıştırmak için: pip install flask opencv-python")
print("Sonra: python app.py")
print("Tarayıcıda: http://localhost:5000")

---
# 🏛️ BÖLÜM 9: Alıştırmalar

Aşağıdaki sorular yalnızca bu haftanın konularını değil, bilgisayar bilimi, matematik ve mühendislik perspektifini birleştirmenizi gerektirir. **Cevaplar verilmemiştir — çözüm sürecini düşünmek öğrenmenin kendisidir.**

---
## ❓ Soru 1 — SSIM Tabanlı Kalite Değerlendirici

**Zorluk: ⭐⭐⭐☆☆**

Farklı interpolasyon yöntemlerinin görsel kalitesini **nesnel** olarak ölçen bir program yazın.

### Görev:
1. Bir görüntüyü %50 oranında küçültün (orijinal → küçük)
2. Küçültülmüş görüntüyü tekrar orijinal boyutuna büyütün (küçük → yeniden büyütülmüş)
3. Orijinal ile yeniden büyütülmüş görüntüyü karşılaştırın
4. Her interpolasyon yöntemi için şu metrikleri hesaplayın:
   - **MSE** (Mean Squared Error): Ortalama karesel hata
   - **PSNR** (Peak Signal-to-Noise Ratio): `20 × log10(255 / sqrt(MSE))`
   - **SSIM** (Structural Similarity Index): `cv2.quality.QualitySSIM_compute()`
5. Sonuçları bir tablo ve grafik olarak görselleştirin

### Düşünmeniz Gereken Sorular:
- Küçük bir görüntüyü büyüttükten sonra hiçbir zaman orijinal kalitesine ulaşılamaz — neden?
- PSNR yüksek olmasına rağmen görüntü kötü görünebilir mi? (Evet! Neden?)
- SSIM insanın görsel algısına neden PSNR'dan daha yakındır?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def hesapla_psnr(orijinal, karsilastirilan):
    # TODO
    pass

def karsilastir_interpolasyonlar(resim_yolu):
    # TODO
    pass
```

---
## ❓ Soru 2 — Otomatik Belge Tarayıcı (Document Scanner)

**Zorluk: ⭐⭐⭐⭐☆**

Eğri tutulmuş veya perspektif bozukluğu olan bir kağıt/belge fotoğrafını otomatik olarak düzleştiren ve A4 formatına getiren bir program yazın.

### Görev:
1. Görüntüyü gri tonlamaya ve Gaussian blur'a çevirin
2. Canny edge detection ile kenarları bulun
3. `cv.findContours()` ile konturları tespit edin
4. Alan hesaplamasıyla en büyük 4 köşeli konturu bulun (`cv.approxPolyDP()`)
5. 4 köşe noktasını sıralayın (sol-üst, sağ-üst, sağ-alt, sol-alt)
6. `cv.getPerspectiveTransform()` ile perspektif matrisini hesaplayın
7. `cv.warpPerspective()` ile belgeyi düzleştirin

### Düşünmeniz Gereken Sorular:
- Affine dönüşüm ile perspektif dönüşüm arasındaki matematiksel fark nedir? (İpucu: kaç nokta gerekir? Hangi özellik korunur?)
- Gerçek dünya fotoğraflarında köşe tespiti neden zorlaşır?
- Bu algoritmanın CamScanner gibi uygulamaların temelini oluşturduğunu biliyor muydunuz?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def belge_tara(resim_yolu, hedef_genislik=595, hedef_yukseklik=842):  # A4 @ 72dpi
    # TODO: Adımları gerçekleştirin
    pass
```

---
## ❓ Soru 3 — Görüntü Piramidi ve Gaussian/Laplacian Piramit

**Zorluk: ⭐⭐⭐⭐☆**

Görüntü piramitleri, nesne tespiti ve görüntü harmanlamanın temel yapı taşıdır. Bu soruda hem Gaussian hem Laplacian piramidini sıfırdan inşa edeceksiniz.

### Görev:
1. **Gaussian Piramidi:** Görüntüyü 5 seviye boyunca `cv.pyrDown()` ile küçülterek her seviyeyi kaydedin
2. **Laplacian Piramidi:** Her Gaussian seviyesinden bir alt seviyeyi `cv.pyrUp()` ile büyütüp çıkararak Laplacian piramidini oluşturun
   `L[i] = G[i] - pyrUp(G[i+1])`
3. **Yeniden yapılandırma:** Laplacian piramidinden orijinal görüntüyü geri kurtarın
4. **Görüntü harmanlama:** İki farklı görüntüyü Laplacian piramidi kullanarak soldan sağa doğal bir geçişle birleştirin (mask ile)

### Düşünmeniz Gereken Sorular:
- Laplacian piramidi orijinal görüntüyü tam olarak geri üretebilir mi? Neden?
- Basit copy-paste yerine piramit tabanlı harmanlama neden daha doğal görünür?
- Bu teknik JPEG sıkıştırmada nasıl kullanılır? (İpucu: wavelet dönüşümü ile ilişki)

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def gaussian_piramit(image, seviye=5):
    # TODO
    pass

def laplacian_piramit(gaussian_pyr):
    # TODO
    pass

def piramit_harmonla(img1, img2, mask):
    # TODO
    pass
```

---
## ❓ Soru 4 — Gerçek Zamanlı Görüntü Stabilizasyon

**Zorluk: ⭐⭐⭐⭐⭐**

Titreyen/sallanarak çekilen bir videoyu stabilize eden, yani kamera sarsıntısını gidererek akıcı görüntü üreten bir program yazın. Bu, akıllı telefon kameralarındaki "video stabilizasyon" özelliğinin temelidir.

### Görev:
1. Video karelerini okuyun
2. Ardışık iki kare arasındaki hareketi `cv.calcOpticalFlowPyrLK()` veya `cv.estimateAffinePartial2D()` ile tespit edin
3. Kümülatif dönüşümü (cumulative transform) takip edin
4. Dönüşümleri düzleştirmek için **hareketli ortalama (moving average)** uygulayın
5. Her kareye düzleştirilmiş dönüşümün tersini `warpAffine()` ile uygulayın
6. Orijinal ve stabilize edilmiş videoyu yan yana gösterin

### Düşünmeniz Gereken Sorular:
- Neden kümülatif dönüşüm takibi yerine kare-kare karşılaştırma yeterli değildir?
- Moving average window boyutu büyüdükçe ne olur? Tradeoff nedir?
- Optical Image Stabilization (OIS — fiziksel lens hareketi) ile EIS (Electronic — bu yöntem) farkı nedir? Hangisi daha iyi ve neden?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np

def stabilize_et(video_yolu, cikti_yolu, pencere=30):
    """
    pencere: moving average için kare sayısı
    Büyük pencere = daha fazla düzlük ama daha fazla kırpma
    """
    # TODO
    pass
```

---
## ❓ Soru 5 — Eğitim Verisi Artırma (Data Augmentation) Pipeline'ı

**Zorluk: ⭐⭐⭐⭐⭐**

Derin öğrenme modellerini eğitmek için kullanılan, tek bir görüntüden onlarca farklı varyant üreten gerçekçi bir augmentation sistemi tasarlayın ve uygulayın.

### Görev:
Aşağıdaki augmentation işlemlerini **parametreli ve rastgele** olarak uygulayan bir sınıf yazın:

| Augmentation | Parametre Aralığı |
|--------------|------------------|
| Rastgele döndürme | ±15° |
| Rastgele ölçekleme | 0.8× – 1.2× |
| Rastgele yatay flip | %50 olasılık |
| Rastgele kırpma | Boyutun %80-100'ü |
| Parlaklık değişimi | ±30 |
| Kontrast değişimi | 0.8× – 1.2× |
| Gaussian gürültü | σ = 0-10 |
| Rastgele silme (cutout) | Görüntünün %10-20'si |

**Ek gereksinim:** Görüntüyle birlikte bounding box koordinatları da (nesne tespiti etiketleri) aynı dönüşümlere tabi tutulmalıdır. Flip yapıldığında bounding box da mirror'lanmalı, döndürme yapıldığında da koordinatlar dönüşüm matrisiniz uygulanmalıdır.

### Düşünmeniz Gereken Sorular:
- Neden bounding box augmentasyonu görüntü augmentasyonundan daha zordur?
- Albumentations ve torchvision.transforms kütüphaneleri bu problemi nasıl çözmüştür?
- Aşırı augmentation modeli nasıl etkiler? (İpucu: underfitting)
- Tıbbi görüntülemede augmentation tasarımı neden özellikle dikkat gerektirir?

```python
# Başlangıç iskeleti
import cv2 as cv
import numpy as np
from dataclasses import dataclass
from typing import Optional, Tuple

@dataclass
class BoundingBox:
    x1: float  # normalize (0-1)
    y1: float
    x2: float
    y2: float
    sinif_id: int

class AugmentationPipeline:
    def __init__(self, seed: Optional[int] = None):
        # TODO: Parametreleri ve random state'i başlatın
        pass

    def uygula(self, image: np.ndarray,
               boxes: list[BoundingBox] = None) -> Tuple[np.ndarray, list]:
        """
        Görüntüye ve bounding box'lara aynı rastgele dönüşümleri uygular.
        Returns: (augmented_image, augmented_boxes)
        """
        # TODO
        pass
```

---
<div style="background: linear-gradient(135deg, #0a0a0a 0%, #1a1a2e 40%, #2d1b69 100%);padding: 38px 42px; border-radius: 18px; text-align: center; margin-top: 20px; border: 1px solid #7b2fff44;"><h2 style="color: #7b2fff; font-family: 'Courier New', monospace; margin: 0 0 22px 0;">🎯 8. Hafta Özeti</h2><div style="color: #cdd6f4; text-align: left; max-width: 700px; margin: 0 auto; line-height: 2.2;">✅ <b>İnterpolasyon:</b> NEAREST / LINEAR / CUBIC / LANCZOS4 / AREA — hangisi ne zaman?<br>✅ <code>cv.resize()</code>: boyutunu belirtin <b>(genişlik, yükseklik)</b> sırasında<br>✅ NumPy <code>[::n, ::n]</code>: hızlı ama kaba küçültme alternatifi<br>✅ <code>cv.flip(0/1/-1)</code>: yatay, dikey veya her iki eksen aynalama<br>✅ <code>cv.rotate()</code>: 90°/180° için hızlı yol<br>✅ <code>cv.getRotationMatrix2D()</code> + <code>cv.warpAffine()</code>: serbest açı döndürme<br>✅ Affine dönüşüm matrisi: öteleme + ölçekleme + döndürme + kesme hepsi tek matris<br>✅ Batch resize: glob + Path ile toplu görüntü pipeline'ı<br>✅ Veri artırma (augmentation): flip + rotate + resize'ın makine öğrenmesindeki rolü</div><p style="color: #6b7280; margin: 22px 0 0 0; font-family: 'Courier New', monospace;">9. Hafta → Kenar Tespiti, Konturlar ve Şekil Analizi 🔍</p></div>